# Similarity Search · Find & Group Photos by Visual Style

**How it works:**
1. **qwen3.5** looks at each photo and writes a rich description of its visual style — lighting, mood, composition, tonal character
2. **nomic-embed-text** converts that description into a list of 768 numbers (an *embedding* — a numeric fingerprint of the photo's character)
3. Embeddings are stored in DuckDB alongside the photos
4. To find similar photos: compare fingerprints using *cosine similarity* (1.0 = identical character, 0.0 = nothing in common)
5. To group photos: K-means clustering sorts all fingerprints into buckets — photos in the same bucket look alike

**Prerequisites:**
```bash
ollama pull nomic-embed-text   # ~274MB, one-time download
```

## Config & Imports

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

# ── config ────────────────────────────────────────────────────
VL_MODEL        = "qwen3.5:9b"        # vision model for style descriptions
EMBED_MODEL     = "nomic-embed-text"  # embedding model (run: ollama pull nomic-embed-text)
N_CLUSTERS      = 3                   # number of visual groups for clustering
EMBED_DIM       = 768                 # nomic-embed-text output dimension
PROCESS_PHOTO   = "All"               # filename (e.g. "Kinan.Sweidan-1.jpg") or "All"

# ── imports ───────────────────────────────────────────────────
import base64, json, time, urllib.request
from pathlib import Path

import duckdb
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from IPython.display import display as ipy_display, HTML

RAW_DIR = Path("./data/raw")
photos = (
    sorted(p for ext in ("*.jpg", "*.JPG") for p in RAW_DIR.glob(ext))
    if PROCESS_PHOTO == "All"
    else [RAW_DIR / PROCESS_PHOTO]
)
print(f"Photos: {[p.name for p in photos]}")

## Database Setup

In [ ]:
con = duckdb.connect("./outputs/photos.duckdb")

con.execute(f"""
    CREATE TABLE IF NOT EXISTS photo_embeddings (
        photo_id        VARCHAR PRIMARY KEY,
        image_path      VARCHAR NOT NULL,
        style_desc      TEXT,
        embedding       FLOAT[{EMBED_DIM}],
        created_at      TIMESTAMP DEFAULT now()
    )
""")

count = con.execute("SELECT count(*) FROM photo_embeddings").fetchone()[0]
print(f"photo_embeddings table ready · {count} rows stored")

## Helpers · Describe & Embed

In [ ]:
STYLE_PROMPT = """You are a photography expert. Look at this black and white street photograph and describe its visual and artistic character in one dense paragraph.

Cover all of these — be specific, not generic:
- SUBJECT: Who or what is the primary subject? Posture, clothing texture, expression.
- COMPOSITION: Where is the subject in the frame? Rule of thirds, symmetry, leading lines, negative space.
- LIGHTING: Direction (front / side / backlit), quality (harsh vs soft), highlight placement, shadow depth.
- TONAL RANGE: High-contrast (deep blacks, bright whites) or low-contrast (flat, grey midtones)? Describe the overall key — low-key (predominantly dark), high-key (predominantly light), or midtone.
- MOOD: What emotional atmosphere does the image project? (e.g. melancholic, tense, contemplative, joyful)
- STYLE: What photographic tradition does it recall? (e.g. documentary, fine art, surrealist, humanist)

Write only the paragraph. No field labels. No commentary. No reasoning."""


def img_b64(path: Path) -> str:
    """Read image file and return base64-encoded string."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


def describe_photo(path: Path) -> str:
    """Call qwen3.5 to produce a rich visual style description of the photo."""
    payload = {
        "model": VL_MODEL,
        "prompt": STYLE_PROMPT,
        "images": [img_b64(path)],
        "stream": False,
        "think": False,
    }
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.loads(r.read())["response"].strip()


def get_embedding(text: str) -> list[float]:
    """Convert text to a 768-dim float vector using nomic-embed-text."""
    payload = {"model": EMBED_MODEL, "input": text}
    req = urllib.request.Request(
        "http://localhost:11434/api/embed",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.loads(r.read())["embeddings"][0]  # list of 768 floats


def photo_id(path: Path) -> str:
    """Stable ID: SHA-256 of first 64KB of file (first 16 hex chars)."""
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(65536))
    return h.hexdigest()[:16]


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors. 1.0 = identical, 0.0 = unrelated."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


print("Helpers ready.")

## Ingest · Describe + Embed All Photos

Run this once. Already-processed photos are skipped automatically.

In [ ]:
for path in photos:
    pid = photo_id(path)

    # Skip if already in DB
    exists = con.execute(
        "SELECT 1 FROM photo_embeddings WHERE photo_id = ?", [pid]
    ).fetchone()
    if exists:
        print(f"  skip  {path.name}  (already stored)")
        continue

    # Step 1: qwen3.5 describes the visual style
    t0 = time.time()
    desc = describe_photo(path)
    t1 = time.time()

    # Step 2: nomic-embed-text converts description → 768 numbers
    emb = get_embedding(desc)
    t2 = time.time()

    # Step 3: Store in DuckDB
    con.execute(
        """
        INSERT INTO photo_embeddings (photo_id, image_path, style_desc, embedding)
        VALUES (?, ?, ?, ?)
        """,
        [pid, str(path.resolve()), desc, emb],
    )

    print(f"  ✓  {path.name}  describe:{t1-t0:.1f}s  embed:{t2-t1:.1f}s")

total = con.execute("SELECT count(*) FROM photo_embeddings").fetchone()[0]
print(f"\nDone · {total} photos in photo_embeddings table")

## Find Similar Photos

Change `QUERY_PHOTO` to any filename in `data/raw/` and run this cell.

In [ ]:
QUERY_PHOTO = "Kinan.Sweidan-1.jpg"   # ← change this to any photo
TOP_N       = 4                        # how many similar photos to return

query_path = RAW_DIR / QUERY_PHOTO
query_pid  = photo_id(query_path)

# Get query embedding from DB (or generate if missing)
row = con.execute(
    "SELECT embedding, style_desc FROM photo_embeddings WHERE photo_id = ?",
    [query_pid],
).fetchone()

if row is None:
    print(f"'{QUERY_PHOTO}' not yet ingested — run the Ingest cell first.")
else:
    q_emb  = np.array(row[0], dtype=np.float32)
    q_desc = row[1]

    # Load all other photos and compute cosine similarity
    rows = con.execute(
        "SELECT photo_id, image_path, style_desc, embedding FROM photo_embeddings WHERE photo_id != ?",
        [query_pid],
    ).fetchall()

    scored = [
        (cosine_sim(q_emb, np.array(r[3], dtype=np.float32)), r[1], r[2])
        for r in rows
    ]
    scored.sort(reverse=True)
    top = scored[:TOP_N]

    # ── Display ────────────────────────────────────────────────
    ncols = TOP_N + 1
    fig, axes = plt.subplots(1, ncols, figsize=(3.5 * ncols, 5))

    # Query photo (leftmost)
    axes[0].imshow(Image.open(query_path), cmap="gray")
    axes[0].set_title(f"QUERY\n{QUERY_PHOTO}", fontsize=8, color="#333", fontweight="bold")
    axes[0].axis("off")
    # Divider line to visually separate query from results
    axes[0].axvline(x=Image.open(query_path).width, color="#ccc", linewidth=2)

    for i, (score, img_path, desc) in enumerate(top, start=1):
        name = Path(img_path).name
        axes[i].imshow(Image.open(img_path), cmap="gray")
        axes[i].set_title(f"#{i}  similarity: {score:.3f}\n{name}", fontsize=8, color="#555")
        axes[i].axis("off")

    plt.suptitle(f"Photos most similar to '{QUERY_PHOTO}'", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

    # Print descriptions for context
    ipy_display(HTML(f"""
    <div style="font-family:monospace; font-size:11px; max-width:1100px">
      <div style="margin-bottom:12px">
        <strong>QUERY · {QUERY_PHOTO}</strong><br>
        <span style="color:#555">{q_desc}</span>
      </div>
      {''.join(
        f'<div style="margin-bottom:10px; padding:8px; background:#f9f9f9; border-left:3px solid #ddd">'
        f'<strong>#{i+1} · {Path(ip).name} · similarity {sc:.3f}</strong><br>'
        f'<span style="color:#555">{dc}</span></div>'
        for i, (sc, ip, dc) in enumerate(top)
      )}
    </div>
    """))

## Visual Grouping · Cluster Photos by Style

K-means groups all photos into `N_CLUSTERS` buckets. Photos in the same bucket have the most in common visually. Try changing `N_CLUSTERS` (top of notebook) and re-running.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

# Load all embeddings + paths
rows = con.execute(
    "SELECT photo_id, image_path, embedding FROM photo_embeddings ORDER BY photo_id"
).fetchall()

if len(rows) < N_CLUSTERS:
    print(f"Need at least {N_CLUSTERS} photos ingested. Run Ingest cell first.")
else:
    pids   = [r[0] for r in rows]
    paths  = [r[1] for r in rows]
    matrix = np.array([r[2] for r in rows], dtype=np.float32)  # shape: (n_photos, 768)

    # Normalize before K-means so it respects cosine similarity
    matrix_normed = normalize(matrix)

    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
    labels = km.fit_predict(matrix_normed)

    # Group photo paths by cluster label
    clusters = {i: [] for i in range(N_CLUSTERS)}
    for label, path in zip(labels, paths):
        clusters[label].append(path)

    # ── Display each cluster as a photo grid ──────────────────
    for cluster_id, cluster_paths in sorted(clusters.items()):
        n = len(cluster_paths)
        fig, axes = plt.subplots(1, n, figsize=(3 * n, 4))
        if n == 1:
            axes = [axes]

        for ax, img_path in zip(axes, cluster_paths):
            ax.imshow(Image.open(img_path), cmap="gray")
            ax.set_title(Path(img_path).name, fontsize=7, color="#555")
            ax.axis("off")

        fig.suptitle(f"Group {cluster_id + 1} · {n} photo{'s' if n != 1 else ''}",
                     fontsize=11, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.show()

## Inspect Stored Descriptions

Quick look at what qwen3.5 wrote for each photo.

## Export · HTML

In [ ]:
import subprocess, sys, datetime
from pathlib import Path

out_dir = Path("./outputs/html")
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / f"similarity_search_{datetime.date.today()}.html"

result = subprocess.run(
    [sys.executable, "-m", "nbconvert", "--to", "html", "--no-input",
     "similarity_search.ipynb", "--output", str(out_file.resolve())],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"✓ Exported to {out_file}")
else:
    print(result.stderr)

In [ ]:
rows = con.execute(
    "SELECT image_path, style_desc, created_at FROM photo_embeddings ORDER BY image_path"
).fetchall()

html_parts = []
for img_path, desc, created_at in rows:
    name = Path(img_path).name
    html_parts.append(
        f'<div style="margin-bottom:16px; font-family:monospace; font-size:11px; max-width:800px">'
        f'<strong>{name}</strong> '
        f'<span style="color:#aaa; font-size:10px">{created_at}</span><br>'
        f'<span style="color:#444">{desc}</span>'
        f'</div>'
    )

ipy_display(HTML("".join(html_parts)))